# Stable Virtual Camera API - Colab Demo

This notebook demonstrates the Stable Virtual Camera API for novel view synthesis. The API extracts core model functionality from the original Gradio demo and provides clean, modular endpoints for programmatic access.

- **Direct Python API**: Clean SevaAPI class for programmatic access
- **REST API Server**: FastAPI endpoints for web integration
- **Multiple Trajectories**: Support for orbit, spiral, zoom, and linear camera movements
- **GPU Acceleration**: Optimized for CUDA execution

- GPU runtime (T4, V100, or A100 recommended)
- ~8GB GPU memory for default settings
- Hugging Face token for model access

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryoosk/stable-virtual-camera/blob/devin/1748692706-api-implementation/stable_virtual_camera_api_colab.ipynb)

## 1. Setup and Installation

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available. Please enable GPU runtime: Runtime > Change runtime type > Hardware accelerator > GPU")

In [ ]:
# Clone the repository with API implementation
!git clone https://github.com/ryoosk/stable-virtual-camera.git
%cd stable-virtual-camera
!git checkout devin/1748692706-api-implementation

In [ ]:
# Comprehensive dependency fix for torch ecosystem compatibility

print("🔧 Fixing torch ecosystem compatibility...")

!pip uninstall -y torch torchvision torchaudio transformers diffusers accelerate xformers -q

!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.37.2 diffusers==0.25.1 accelerate==0.26.1

print("✅ Torch ecosystem reinstalled. Restarting runtime...")

import os
os.kill(os.getpid(), 9)

In [ ]:
# Post-restart: Verify imports and complete setup
print("🔍 Verifying torch ecosystem imports...")

try:
    import torch
    import torchvision
    print(f"✅ Torch {torch.__version__}, Torchvision {torchvision.__version__}")
except ImportError as e:
    print(f"❌ Torch import failed: {e}")

try:
    from transformers import AutoImageProcessor
    print("✅ AutoImageProcessor import successful!")
except ImportError as e:
    print(f"❌ AutoImageProcessor import failed: {e}")
    
try:
    from diffusers.models import AutoencoderKL
    print("✅ AutoencoderKL import successful!")
except ImportError as e:
    print(f"❌ AutoencoderKL import failed: {e}")

print("📦 Installing remaining dependencies...")
!pip install -e .
!pip install fastapi uvicorn python-multipart imageio[ffmpeg]

print("✅ All dependencies installed successfully!")

## 2. Authentication and Model Download

You need a Hugging Face token to access the gated model. Get one at: https://huggingface.co/settings/tokens

In [ ]:
import os
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token: ")
os.environ['HUGGINGFACE_TOKEN'] = hf_token
print("✅ Token set successfully!")

In [ ]:
# Download the model (5.06GB)
from huggingface_hub import hf_hub_download

print("Downloading Stable Virtual Camera model...")
try:
    hf_hub_download(
        repo_id='stabilityai/stable-virtual-camera',
        filename='model.safetensors',
        local_dir='.',
        token=os.environ['HUGGINGFACE_TOKEN']
    )
    print("✅ Model downloaded successfully!")
except Exception as e:
    print(f"❌ Download failed: {e}")
    print("Please check your token and try again.")

## 3. Direct API Usage Demo

In [ ]:
# Initialize the API
from seva.api import SevaAPI
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch

print("Initializing Stable Virtual Camera API...")
api = SevaAPI(device="cuda", compile_model=False)
print("✅ API initialized successfully!")

In [ ]:
# Test image preprocessing
test_image_path = "assets/basic/vasedeck.jpg"

print(f"Testing preprocessing with: {test_image_path}")
preprocessed = api.preprocess_single_image(test_image_path)

print(f"✅ Preprocessing successful!")
print(f"Input image shape: {preprocessed['input_imgs'].shape}")
print(f"Camera intrinsics shape: {preprocessed['input_Ks'].shape}")
print(f"Camera pose shape: {preprocessed['input_c2ws'].shape}")
print(f"Image dimensions: {preprocessed['input_wh']}")

input_img = preprocessed['input_imgs'][0].numpy()
plt.figure(figsize=(8, 6))
plt.imshow(input_img)
plt.title("Input Image")
plt.axis('off')
plt.show()

In [ ]:
# Generate novel views with orbit trajectory
print("Generating novel views with orbit trajectory...")
print("This may take 2-5 minutes depending on GPU...")

results = api.render_novel_views(
    preprocessed_data=preprocessed,
    trajectory_type="orbit",
    num_frames=8,  # Reduced for faster demo
    cfg_scale=3.0,
    seed=42
)

rendered_images = results["rendered_images"]
print(f"✅ Generated {len(rendered_images)} frames!")

In [ ]:
# Display results
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, img_tensor in enumerate(rendered_images):
    if isinstance(img_tensor, torch.Tensor):
        img_np = img_tensor.cpu().numpy()
    else:
        img_np = img_tensor
    
    if img_np.max() <= 1.0:
        img_np = (img_np * 255).astype(np.uint8)
    
    axes[i].imshow(img_np)
    axes[i].set_title(f"Frame {i+1}")
    axes[i].axis('off')

plt.suptitle("Novel View Synthesis - Orbit Trajectory", fontsize=16)
plt.tight_layout()
plt.show()

## 4. Different Trajectory Types Demo

In [ ]:
# Test different trajectory types
trajectory_types = ["spiral", "zoom-in", "move-forward"]
trajectory_results = {}

for traj_type in trajectory_types:
    print(f"Generating {traj_type} trajectory...")
    
    results = api.render_novel_views(
        preprocessed_data=preprocessed,
        trajectory_type=traj_type,
        num_frames=4,  # Fewer frames for demo
        cfg_scale=3.0,
        seed=42
    )
    
    trajectory_results[traj_type] = results["rendered_images"]
    print(f"✅ {traj_type} completed!")

print("All trajectories generated!")

In [ ]:
# Display trajectory comparison
fig, axes = plt.subplots(len(trajectory_types), 4, figsize=(16, 12))

for row, traj_type in enumerate(trajectory_types):
    for col, img_tensor in enumerate(trajectory_results[traj_type]):
        if isinstance(img_tensor, torch.Tensor):
            img_np = img_tensor.cpu().numpy()
        else:
            img_np = img_tensor
        
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype(np.uint8)
        
        axes[row, col].imshow(img_np)
        if col == 0:
            axes[row, col].set_ylabel(traj_type, fontsize=12)
        axes[row, col].set_title(f"Frame {col+1}")
        axes[row, col].axis('off')

plt.suptitle("Trajectory Comparison", fontsize=16)
plt.tight_layout()
plt.show()

## 5. Upload Your Own Image

In [ ]:
# Upload your own image
from google.colab import files
import os

print("Upload an image file:")
uploaded = files.upload()

if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    print(f"Uploaded: {uploaded_filename}")
    
    print("Processing uploaded image...")
    custom_preprocessed = api.preprocess_single_image(uploaded_filename)
    
    custom_results = api.render_novel_views(
        preprocessed_data=custom_preprocessed,
        trajectory_type="orbit",
        num_frames=6,
        cfg_scale=3.0,
        seed=42
    )
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, img_tensor in enumerate(custom_results["rendered_images"]):
        if isinstance(img_tensor, torch.Tensor):
            img_np = img_tensor.cpu().numpy()
        else:
            img_np = img_tensor
        
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype(np.uint8)
        
        axes[i].imshow(img_np)
        axes[i].set_title(f"Frame {i+1}")
        axes[i].axis('off')
    
    plt.suptitle(f"Custom Image Results: {uploaded_filename}", fontsize=16)
    plt.tight_layout()
    plt.show()
    
    print("✅ Custom image processing completed!")
else:
    print("No file uploaded.")

## 6. Summary and Next Steps

🎉 **Congratulations!** You've successfully tested the Stable Virtual Camera API!

- ✅ Set up the API in a GPU environment
- ✅ Tested image preprocessing and novel view synthesis
- ✅ Explored different camera trajectories
- ✅ Processed custom images

- **Trajectory Types**: orbit, spiral, lemniscate, zoom-in/out, dolly zoom, linear movements
- **Flexible Interface**: Direct Python API and REST endpoints
- **GPU Optimized**: CUDA acceleration for fast inference
- **Modular Design**: Clean separation from UI dependencies

1. **Integration**: Use the API in your own applications
2. **Customization**: Modify trajectory parameters and camera settings
3. **Scaling**: Deploy the REST API server for production use
4. **Enhancement**: Contribute to the project with new features

- **GitHub Repository**: https://github.com/ryoosk/stable-virtual-camera
- **API Documentation**: See `API_README.md` in the repository
- **Original Paper**: Stable Virtual Camera for Novel View Synthesis

Happy coding! 🚀